# Maize Disease Detection using Neural Networks

## 1. Problem Statement

Maize foliar diseases — **Common Rust**, **Gray Leaf Spot** and **Blight** — spread rapidly and can cut yields by 30–50%. The current diagnosis method is visual inspection by a human agronomist: slow, subjective, error-prone, and impossible to scale across large plantations. Many smallholder farmers have no access to a plant pathologist, so disease is caught only after irreversible damage.

**Objective:** Train a CNN to classify a maize leaf image into one of four classes — *Healthy, Common Rust, Gray Leaf Spot, Blight* — so a farmer can photograph a leaf and get an instant diagnosis.

**Research questions:**
1. Which CNN architecture (custom CNN, VGG16, ResNet50, InceptionV3, MobileNetV2) gives the best accuracy, and how do model size and training time trade off against it?
2. How do augmentation and class-balancing affect generalisation, especially recall on the minority Gray Leaf Spot class?
3. Which disease pairs are most confused, and what causes it — background clutter, lighting, or the visual overlap between Common Rust and Blight?

## 2. Project Overview

A supervised image-classification pipeline:

```
Data → Cleaning → EDA → Train/Val/Test Split → Augmentation → CNN Training
     → Evaluation → Error Analysis → Error Reduction → Prediction
```

A CNN learns features directly from raw pixels — edges and colour blobs in early layers, lesion shapes and textures in deeper ones — so no hand-crafted features are needed. The dataset is imbalanced (Gray Leaf Spot has ~43% fewer images than Common Rust), so class weights and macro-averaged metrics are used to stop the model favouring majority classes.

## 3. Dataset Overview

**Source:** [Kaggle — Corn or Maize Leaf Disease Dataset](https://www.kaggle.com/datasets/smaranjitghose/corn-or-maize-leaf-disease-dataset)

| Class | Images | Share |
|---|---|---|
| Common_Rust | 1,306 | 31.2% |
| Healthy | 1,162 | 27.7% |
| Blight | 1,146 | 27.4% |
| Gray_Leaf_Spot | 574 | 13.7% |
| **Total** | **4,188** | **100%** |

Real-world leaf photographs stored locally as `data/<Class_Name>/*.jpg`; labels come from the folder names. **Known issues in the raw data:** mixed file extensions (`.jpg`/`.JPG`/`.jpeg`), varying resolutions, non-uniform backgrounds and lighting, possible duplicate captures, and label noise where harmless blemishes resemble early-stage disease.

## 4. Setup and Imports

In [2]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")